# Nested Pandas
Demo at LSST DM meeting, 2026-09-16

Run on the RSP using the latest weekly environment.

https://nested-pandas.rtfd.io, https://docs.lsdb.io

In [ ]:
import lsdb
import nested_pandas as npd
import numpy as np

In [ ]:
df = lsdb.open_catalog(
    "/rubin/lsdb_data/dp2/dia_object_collection/",
    search_filter=lsdb.ConeSearch(151.16186, 2.33994, radius_arcsec=20),
    columns = ["diaObjectForcedSource.band", "diaObjectForcedSource.midpointMjdTai",
              "diaObjectForcedSource.psfFlux", "diaObjectForcedSource.psfDiffFlux",
              "diaObjectForcedSource.psfFluxErr", "diaObjectForcedSource.psfDiffFluxErr",
              "diaSource.band", "diaSource.psfFlux", "diaSource.psfFluxErr",
              "diaSource.midpointMjdTai"]
).compute()

In [ ]:
df

In [ ]:
# Access single nested frame
df.iloc[0]["diaObjectForcedSource"]

In [ ]:
# Use NestedSeries methods
df["diaObjectForcedSource"].len()

In [ ]:
# filter by length
df.query("diaSource.len() > 1")
# same as
df[df["diaSource"].len() > 1]

In [ ]:
# Filter *nested* rows, this does not filter *base* rows
df.query("abs(diaSource.psfFlux / diaSource.psfFluxErr) > 5")

In [ ]:
# Flat subcolumn representation
df["diaObjectForcedSource.psfDiffFlux"]

In [ ]:
# Flat subcolumn assignment
df["diaObjectForcedSource.mag"] = 31.4 - 2.5 * np.log10(df["diaObjectForcedSource.psfFlux"])
df.query("24 > diaObjectForcedSource.mag > 21")

In [ ]:
# Per-row user function, similar to .apply

def chi2(row):
    average = np.average(row["diaSource.psfFlux"], weights=row["diaSource.psfFluxErr"]**-2)
    return np.sum((row["diaSource.psfFlux"] - average) / row["diaSource.psfFluxErr"])**2

df = df.map_rows(chi2, output_names=["chi2"], append_columns=True)
df

In [ ]:
# Per-row user function, positional arguments
duration = df.map_rows(np.ptp, row_container="args", columns=["diaObjectForcedSource.midpointMjdTai"])[0]
duration

In [ ]:
# some useful NestedFrame methods
# Convert to "flat" source table
flat_sources = df[["ra", "dec", "diaSource"]].explode("diaSource")
flat_sources

In [ ]:
# Pack back
new_df = npd.NestedFrame.from_flat(flat_sources, base_columns=["ra", "dec"], name="dia_sources")

In [ ]:
# Split into single-band light curves
single_band_fourced_sources = df.split("diaObjectForcedSource", "band", values="ugrizy")
single_band_fourced_sources